# ECG GAN - Unified Notebook

This notebook combines preprocessing, model definition, training, saving, loading, and evaluation into one flow.  
It keeps the original GAN idea, but organizes everything so the notebook can run section by section more clearly.

## 1. Imports and setup

This section loads the libraries, sets the random seed, and prepares the device and main settings used across the notebook.

In [ ]:

import os
import time
import random
import math

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [ ]:

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Configuration

This section keeps the main paths and hyperparameters in one place so it is easier to edit later.

In [ ]:

data_dir = "./training2017"
processed_dir = "./processed_data"
checkpoint_dir = "./checkpoints"

seq_len = 3000
sampling_rate = 300
batch_size = 32
train_ratio = 0.8
seed = 42

noise_dim = 5
hidden_dim = 256
gen_dropout = 0.3

num_epochs = 50
lr_g = 1e-5
lr_d = 1e-5
beta1 = 0.5
beta2 = 0.999

real_low = 0.85
real_high = 0.95
fake_low = 0.00
fake_high = 0.15

noise_std_start = 0.05
noise_std_end = 0.03
extra_g_threshold = 0.65

print_every = 10
save_every = 10

os.makedirs(processed_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

## 3. Preprocessing functions

This section loads the ECG files, applies the bandpass filter, normalizes the signals, and splits them into fixed windows.

In [ ]:

def load_ecg(file_path):
    mat = sio.loadmat(file_path)
    return mat["val"].flatten().astype(np.float64)

def bandpass_filter(signal, lowcut=0.5, highcut=40.0):
    nyquist = sampling_rate / 2.0
    b, a = butter(4, [lowcut / nyquist, highcut / nyquist], btype="band")
    return filtfilt(b, a, signal)

def normalize_signal(signal):
    sig_min = signal.min()
    sig_max = signal.max()
    if sig_max - sig_min == 0:
        return np.zeros_like(signal)
    return (signal - sig_min) / (sig_max - sig_min)

def segment_signal(signal, window_len=seq_len, step=None):
    if step is None:
        step = window_len
    segments = []
    start = 0
    while start + window_len <= len(signal):
        segments.append(signal[start:start + window_len])
        start += step
    return segments

def preprocess_data(data_dir, overlap=0.0, allowed_labels=("N", "A", "O")):
    labels = pd.read_csv(
        os.path.join(data_dir, "REFERENCE.csv"),
        header=None,
        names=["recording", "label"]
    )

    selected = labels[labels["label"].isin(allowed_labels)].copy()

    np.random.seed(seed)
    recording_ids = selected["recording"].values
    shuffled_idx = np.random.permutation(len(recording_ids))
    split_idx = int(len(recording_ids) * train_ratio)

    train_ids = set(recording_ids[shuffled_idx[:split_idx]])
    test_ids = set(recording_ids[shuffled_idx[split_idx:]])

    step = int(seq_len * (1 - overlap)) if overlap > 0 else seq_len
    step = max(1, step)

    train_segments = []
    test_segments = []
    skipped = 0

    for i, row in selected.iterrows():
        file_path = os.path.join(data_dir, row["recording"] + ".mat")
        signal = load_ecg(file_path)
        signal = bandpass_filter(signal)
        signal = normalize_signal(signal)
        segments = segment_signal(signal, window_len=seq_len, step=step)

        if len(segments) == 0:
            skipped += 1
            continue

        if row["recording"] in train_ids:
            train_segments.extend(segments)
        else:
            test_segments.extend(segments)

    train_segments = np.array(train_segments, dtype=np.float32)
    test_segments = np.array(test_segments, dtype=np.float32)

    np.random.seed(seed)
    if len(train_segments) > 0:
        train_segments = train_segments[np.random.permutation(len(train_segments))]

    print("train recordings:", len(train_ids))
    print("test recordings:", len(test_ids))
    print("train segments:", len(train_segments))
    print("test segments:", len(test_segments))
    print("skipped recordings:", skipped)

    if len(train_segments) > 0:
        print("train range:", float(train_segments.min()), "to", float(train_segments.max()))

    return train_segments, test_segments

## 4. Run preprocessing and save arrays

This section processes the dataset once and saves the train and test arrays so later sections can use them directly.

In [ ]:

train_data, test_data = preprocess_data(data_dir=data_dir, overlap=0.0)

train_path = os.path.join(processed_dir, "ecg_train.npy")
test_path = os.path.join(processed_dir, "ecg_test.npy")

np.save(train_path, train_data)
np.save(test_path, test_data)

print("saved:", train_path)
print("saved:", test_path)

## 5. Build the training dataloader

This section converts the processed training array into PyTorch tensors and creates the dataloader used by the GAN.

In [ ]:

train_tensor = torch.from_numpy(train_data).float()
if train_tensor.ndim == 2:
    train_tensor = train_tensor.unsqueeze(-1)

train_set = TensorDataset(train_tensor)
train_loader = DataLoader(train_set,batch_size,shuffle=True,drop_last=True)

print("train tensor shape:", train_tensor.shape)
print("batches per epoch:", len(train_loader))

## 6. Model definitions

This section defines the BiLSTM generator and the CNN discriminator used in the original training idea.

In [ ]:

class generator(nn.Module):
    def __init__(self, noise_dim=5, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.bilstm1 = nn.LSTM(
            input_size=noise_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.bilstm2 = nn.LSTM(
            input_size=hidden_dim * 2,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.out = nn.Sigmoid()

    def forward(self, z):
        x, _ = self.bilstm1(z)
        x, _ = self.bilstm2(x)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.out(x)
        return x


class discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 10, kernel_size=120, stride=5)
        self.pool1 = nn.MaxPool1d(kernel_size=46, stride=3)
        self.conv2 = nn.Conv1d(10, 5, kernel_size=36, stride=3)
        self.pool2 = nn.MaxPool1d(kernel_size=24, stride=3)
        self.fc1 = nn.Linear(45, 25)
        self.fc2 = nn.Linear(25, 1)
        self.out = nn.Sigmoid()

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.out(x)
        return x

## 7. Initialize models and training objects

This section creates the models, loss function, optimizers, and the history dictionary for logging.

In [ ]:

gen = generator(noise_dim=noise_dim, hidden_dim=hidden_dim, dropout=gen_dropout).to(device)
disc = discriminator().to(device)

loss_fn = nn.BCELoss()

opt_g = optim.Adam(gen.parameters(), lr=lr_g, betas=(beta1, beta2))
opt_d = optim.Adam(disc.parameters(), lr=lr_d, betas=(beta1, beta2))

history = {
    "loss_d": [],
    "loss_g": [],
    "d_real": [],
    "d_fake_before_g": [],
    "d_fake_after_g": []
}

print(gen)
print(disc)

## 8. Training helper functions

This section groups the small helper functions used during training such as label smoothing and instance noise.

In [ ]:

def make_real_labels(size, device):
    return torch.empty(size, device=device).uniform_(real_low, real_high)

def make_fake_labels(size, device):
    return torch.empty(size, device=device).uniform_(fake_low, fake_high)

def add_input_noise(x, std):
    if std <= 0:
        return x
    x = x + std * torch.randn_like(x)
    return torch.clamp(x, 0.0, 1.0)

def current_noise_std(epoch, total_epochs):
    alpha = (epoch - 1) / max(1, total_epochs - 1)
    return noise_std_start + alpha * (noise_std_end - noise_std_start)

## 9. GAN training loop

This section trains the discriminator and generator together, stores the losses, and saves checkpoints during training.

In [ ]:

train_start = time.time()

for epoch in range(1, num_epochs + 1):
    gen.train()
    disc.train()

    epoch_loss_d = 0.0
    epoch_loss_g = 0.0
    epoch_d_real = 0.0
    epoch_d_fake_before = 0.0
    epoch_d_fake_after = 0.0

    std = current_noise_std(epoch, num_epochs)
    epoch_start = time.time()

    for batch_idx, (real_batch,) in enumerate(train_loader):
        real_x = real_batch.to(device)
        cur_batch = real_x.size(0)

        opt_d.zero_grad()

        real_y = make_real_labels((cur_batch,), device)
        real_x_noisy = add_input_noise(real_x, std)
        out_real = disc(real_x_noisy).view(-1)
        loss_d_real = loss_fn(out_real, real_y)
        loss_d_real.backward()

        d_real = out_real.mean().item()

        z = torch.randn(cur_batch, seq_len, noise_dim, device=device)
        fake_x = gen(z)

        fake_y = make_fake_labels((cur_batch,), device)
        fake_x_noisy = add_input_noise(fake_x.detach(), std)
        out_fake = disc(fake_x_noisy).view(-1)
        loss_d_fake = loss_fn(out_fake, fake_y)
        loss_d_fake.backward()

        d_fake_before = out_fake.mean().item()

        loss_d = loss_d_real + loss_d_fake
        opt_d.step()

        opt_g.zero_grad()

        target_y = make_real_labels((cur_batch,), device)
        fake_x_for_g = add_input_noise(fake_x, std)
        out_gen = disc(fake_x_for_g).view(-1)
        loss_g = loss_fn(out_gen, target_y)
        loss_g.backward()
        opt_g.step()

        d_fake_after = out_gen.mean().item()

        if d_real > extra_g_threshold and d_fake_before < 0.20:
            opt_g.zero_grad()
            z_extra = torch.randn(cur_batch, seq_len, noise_dim, device=device)
            fake_x_extra = gen(z_extra)
            fake_x_extra = add_input_noise(fake_x_extra, std)
            target_y_extra = make_real_labels((cur_batch,), device)
            out_gen_extra = disc(fake_x_extra).view(-1)
            loss_g_extra = loss_fn(out_gen_extra, target_y_extra)
            loss_g_extra.backward()
            opt_g.step()

            loss_g = 0.5 * (loss_g + loss_g_extra)
            d_fake_after = 0.5 * (d_fake_after + out_gen_extra.mean().item())

        epoch_loss_d += loss_d.item()
        epoch_loss_g += float(loss_g)
        epoch_d_real += d_real
        epoch_d_fake_before += d_fake_before
        epoch_d_fake_after += d_fake_after

        if (batch_idx + 1) % print_every == 0:
            print(
                f"epoch [{epoch}/{num_epochs}] "
                f"batch [{batch_idx + 1}/{len(train_loader)}] "
                f"loss_d={loss_d.item():.4f} "
                f"loss_g={float(loss_g):.4f} "
                f"d_real={d_real:.4f} "
                f"d_fake={d_fake_before:.4f}->{d_fake_after:.4f} "
                f"noise_std={std:.4f}"
            )

    n = len(train_loader)
    avg_loss_d = epoch_loss_d / n
    avg_loss_g = epoch_loss_g / n
    avg_d_real = epoch_d_real / n
    avg_d_fake_before = epoch_d_fake_before / n
    avg_d_fake_after = epoch_d_fake_after / n

    history["loss_d"].append(avg_loss_d)
    history["loss_g"].append(avg_loss_g)
    history["d_real"].append(avg_d_real)
    history["d_fake_before_g"].append(avg_d_fake_before)
    history["d_fake_after_g"].append(avg_d_fake_after)

    epoch_time = time.time() - epoch_start

    print(
        f"epoch {epoch:03d} | "
        f"loss_d={avg_loss_d:.4f} | "
        f"loss_g={avg_loss_g:.4f} | "
        f"d_real={avg_d_real:.4f} | "
        f"d_fake={avg_d_fake_before:.4f}->{avg_d_fake_after:.4f} | "
        f"time={epoch_time:.1f}s"
    )

    if epoch % save_every == 0:
        torch.save(gen.state_dict(), os.path.join(checkpoint_dir, f"generator_epoch{epoch}.pth"))
        torch.save(disc.state_dict(), os.path.join(checkpoint_dir, f"discriminator_epoch{epoch}.pth"))
        print("checkpoint saved")

total_time = time.time() - train_start
print("training finished in", total_time / 60, "minutes")

## 10. Save final models

This section saves the final trained generator and discriminator after the full training loop ends.

In [ ]:

gen_path = os.path.join(checkpoint_dir, "generator_final.pth")
disc_path = os.path.join(checkpoint_dir, "discriminator_final.pth")

torch.save(gen.state_dict(), gen_path)
torch.save(disc.state_dict(), disc_path)

print("saved:", gen_path)
print("saved:", disc_path)

## 11. Load models for evaluation

This section reloads the saved generator and discriminator so evaluation can be done separately from training if needed.

In [ ]:

gen_eval = generator(noise_dim=noise_dim, hidden_dim=hidden_dim, dropout=gen_dropout).to(device)
disc_eval = discriminator().to(device)

gen_eval.load_state_dict(torch.load(gen_path, map_location=device))
disc_eval.load_state_dict(torch.load(disc_path, map_location=device))

gen_eval.eval()
disc_eval.eval()

print("models loaded for evaluation")

## 12. Generate ECG samples from the trained generator

This section uses random noise to generate fake ECG signals with the same sequence length as the real test signals.

In [ ]:

if test_data.ndim == 2:
    test_eval = test_data[:, :, np.newaxis]
else:
    test_eval = test_data.copy()

num_samples = len(test_eval)
gen_batch_size = 64
fake_list = []

with torch.no_grad():
    for start in range(0, num_samples, gen_batch_size):
        cur_batch = min(gen_batch_size, num_samples - start)
        z = torch.randn(cur_batch, seq_len, noise_dim, device=device)
        fake_x = gen_eval(z).cpu()
        fake_list.append(fake_x)

fake_data = torch.cat(fake_list, dim=0).numpy()

print("real test shape:", test_eval.shape)
print("fake data shape:", fake_data.shape)

## 13. Evaluation metrics

This section calculates RMSE, PRD, and a discrete Fréchet distance on a smaller subset because that metric is more expensive.

In [ ]:

def rmse_metric(real_x, fake_x):
    real_x = np.squeeze(real_x, axis=-1)
    fake_x = np.squeeze(fake_x, axis=-1)
    per_signal = np.sqrt(np.mean((real_x - fake_x) ** 2, axis=1))
    return float(np.mean(per_signal))

def prd_metric(real_x, fake_x):
    real_x = np.squeeze(real_x, axis=-1)
    fake_x = np.squeeze(fake_x, axis=-1)
    num = np.sum((real_x - fake_x) ** 2, axis=1)
    den = np.maximum(np.sum(real_x ** 2, axis=1), 1e-12)
    return float(np.mean(np.sqrt(num / den) * 100.0))

def downsample_signal(signal, factor=10):
    return signal[::factor]

def discrete_frechet_1d(p, q):
    p = np.array(p)
    q = np.array(q)

    n = len(p)
    m = len(q)
    ca = np.empty((n, m), dtype=np.float64)
    dist = np.abs(p[:, None] - q[None, :])

    ca[0, 0] = dist[0, 0]
    for i in range(1, n):
        ca[i, 0] = max(ca[i - 1, 0], dist[i, 0])
    for j in range(1, m):
        ca[0, j] = max(ca[0, j - 1], dist[0, j])
    for i in range(1, n):
        for j in range(1, m):
            ca[i, j] = max(min(ca[i - 1, j], ca[i - 1, j - 1], ca[i, j - 1]), dist[i, j])

    return float(ca[n - 1, m - 1])

def average_frechet(real_x, fake_x, num_pairs=50, factor=10):
    real_x = np.squeeze(real_x, axis=-1)
    fake_x = np.squeeze(fake_x, axis=-1)
    num_pairs = min(num_pairs, len(real_x))
    scores = []

    for i in range(num_pairs):
        real_ds = downsample_signal(real_x[i], factor=factor)
        fake_ds = downsample_signal(fake_x[i], factor=factor)
        scores.append(discrete_frechet_1d(real_ds, fake_ds))

    return float(np.mean(scores))

In [ ]:

rmse_val = rmse_metric(test_eval, fake_data)
prd_val = prd_metric(test_eval, fake_data)
fd_val = average_frechet(test_eval, fake_data, num_pairs=50, factor=10)

print("rmse:", rmse_val)
print("prd:", prd_val)
print("frechet distance:", fd_val)

## 14. Visualization

This section plots training curves and shows one random real signal against one generated signal for a quick visual check.

In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(history["loss_d"], label="loss_d")
plt.plot(history["loss_g"], label="loss_g")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("training losses")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

sample_idx = np.random.randint(0, num_samples)

real_plot = np.squeeze(test_eval[sample_idx])
fake_plot = np.squeeze(fake_data[sample_idx])

plt.figure(figsize=(12, 4))
plt.plot(real_plot, label="real ecg")
plt.plot(fake_plot, label="generated ecg")
plt.xlabel("time step")
plt.ylabel("amplitude")
plt.title("real vs generated ecg")
plt.legend()
plt.tight_layout()
plt.show()